# Data Loaders

In [ ]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # normalize
        image = image.astype(np.float32) / 255.0

        # create empty mask
        mask = np.zeros(image.shape[:-1], dtype=np.float32)

        label_path = os.path.join(self.label_dir, image_name.replace('.jpg', '.txt'))
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for l in lines:
                parts = l.strip().split()
                class_id = int(parts[0])
                coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)

                coords[:, 0] *= image.shape[0]
                coords[:, 1] *= image.shape[0]
                coords = coords.astype(np.int32)
                cv2.fillPoly(mask, [coords], color=1.0)
        mask = np.expand_dims(mask, axis=-1)           
        return image, mask

        

In [ ]:
def getPotholeDataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)     
    def numpy_collate(batch):
        images, masks = zip(*batch)
        return np.stack(images), np.stack(masks)
        
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=True, 
        drop_last=True, collate_fn=numpy_collate, num_workers=4
    )

In [ ]:
# Test
dataset = PotholeDataset("/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")     
loader = getPotholeDataloader(4, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
dataset[0]

# Rotary Position Embeddings (RoPE)

In [ ]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope(x, head_dim):
    # X -> B,S,H,head_dim
    seq_len = x.shape[1]
    num_heads = x.shape[2]
    dims = jnp.arange(seq_len, dtype=jnp.float32).reshape(-1, 1) # seq_len, 1
    i_2 = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    div_term = jnp.exp(-1* i_2 * jnp.log(100000)/head_dim) # head_dim//2
    freq = dims * div_term # seq_len, head_dim//2
    sin_freq = jnp.repeat(jnp.sin(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    cos_freq = jnp.repeat(jnp.cos(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    odd_terms = x[..., 1::2]
    even_terms = x[..., 0::2]
    x_tmp = jnp.stack([-odd_terms, even_terms], axis=-1).reshape(x.shape)
    return x * cos_freq + x_tmp * sin_freq


In [ ]:
# test
key = jax.random.key(42)
apply_rope(jax.random.normal(key, shape=(5, 10, 12, 20)), 20)

# Multi-Head Self-Attention 

In [ ]:
class MHSA(nn.Module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x):
        # x -> Batch, Seq_len, dim
        b, s, dim = x.shape
        head_dim = dim//self.num_heads

        w_q = self.param('w_q', nn.initializers.lecun_normal(), (dim, dim))
        w_k = self.param('w_k', nn.initializers.lecun_normal(), (dim, dim))
        w_v = self.param('w_v', nn.initializers.lecun_normal(), (dim, dim))
        w_o = self.param('w_o', nn.initializers.lecun_normal(), (dim, dim))

        q = jnp.dot(x, w_q).reshape(b, s, self.num_heads, head_dim)
        k = jnp.dot(x, w_k).reshape(b, s, self.num_heads, head_dim)
        v = jnp.dot(x, w_v).reshape(b, s, self.num_heads, head_dim)

        q = apply_rope(q, head_dim)
        k = apply_rope(k, head_dim)
        
        qk_t = jnp.einsum("bqhd, bkhd -> bhqk", q, k) 
        atten_scores = qk_t/jnp.sqrt(head_dim)
        attention = jax.nn.softmax(atten_scores, axis=-1)
 
        att_out = jnp.einsum('bhqk, bkhd -> bqhd', attention, v).reshape(b, s, dim)
        
        out = jnp.dot(att_out, w_o)
        return out

# Transformer block and adaLN

In [ ]:
class AdaLN(nn.Module):
    dim: int

    @nn.compact
    def __call__(self, x, cond):
        # Zero initializers are important because when we start training we have no idea on scale, shift and gate, so it'll behave as a passthrough
        mlp_out = nn.Dense(self.dim*3, kernel_init=nn.initializers.zeros, bias_init=nn.initializers.zeros)(cond)
        shift, scale, gate = jnp.split(mlp_out, 3, axis=-1)
        x = nn.LayerNorm()(x)

        x = x * (1 + jnp.expand_dims(scale, axis = 1)) + jnp.expand_dims(shift, axis = 1)

        return x, jnp.expand_dims(gate, axis = 1)

In [ ]:
class TransformerBlock(nn.Module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x, global_context):
        # attention path
        dim = x.shape[-1]
        x_norm, mhsa_gate = AdaLN(dim)(x, global_context)
        mhsa_out = MHSA(self.num_heads)(x_norm)        
        x = x + mhsa_gate * mhsa_out

        # MLP path
        x_norm_mlp, mlp_gate = AdaLN(dim)(x, global_context)
        mlp_out = nn.Dense(dim*4)(x_norm_mlp)
        mlp_out = nn.gelu(mlp_out)
        mlp_out = nn.Dense(dim)(mlp_out)
        x = x + mlp_gate * mlp_out
        
        return x      

# Model

In [ ]:
class PotholeSegmenter(nn.Module):
    emb_dim: int = 256
    patch_size: int = 16
    depth: int = 6
    num_heads: int = 8
    
    @nn.compact
    def __call__(self, images):
        B,H,W,C = images.shape
        grid_h = H//self.patch_size
        grid_w = W//self.patch_size

        x = images.reshape((B, grid_h, self.patch_size, grid_w, self.patch_size, C))
        x = jnp.einsum('abcdef -> abdcef',x)
        x = x.reshape((B, grid_h*grid_w, -1))
        
        x = nn.Dense(self.emb_dim)(x)
        global_context = jnp.mean(x, axis=1)

        for _ in range(self.depth):
            x = TransformerBlock(self.num_heads)(x, global_context)

        x = nn.LayerNorm()(x)
        x = nn.Dense(self.patch_size*self.patch_size*1)(x)
        x = x.reshape(B, grid_h, grid_w, self.patch_size, self.patch_size, 1)
        x = jnp.einsum('ahwpqr -> ahpwqr', x)
        logits = x.reshape((B, H, W, 1))
        return logits

# Training 


In [ ]:
import optax
from flax.training import train_state
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from jax.experimental import mesh_utils
from functools import partial
import orbax.checkpoint as ocp

In [ ]:
def bce_with_dice_loss(logits, target):
    bce = optax.sigmoid_binary_cross_entropy(logits, target)
    probs = jax.nn.sigmoid(logits)
    intersection = jnp.sum(probs*target, axis=(1,2,3))
    union = jnp.sum(probs, axis=(1,2,3)) + jnp.sum(target, axis=(1,2,3))
    iou = (2*intersection+1e-5)/(union + 1e-5)
    dice_loss = 1 - jnp.mean(iou)
    return jnp.mean(bce) + dice_loss

In [ ]:
num_devices = jax.device_count()
print(num_devices)
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=('batch',))
print(mesh)

data_sharding = NamedSharding(mesh, PartitionSpec('batch', None, None, None))
param_sharding = NamedSharding(mesh, PartitionSpec())

@partial(
    jax.jit,
    in_shardings=(param_sharding, data_sharding, data_sharding),
    out_shardings=(param_sharding, param_sharding) 
)
def train_step(state, images, masks):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, images)
        loss = bce_with_dice_loss(logits, masks)
        return loss
        
    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss

def train_model():
    batch_size = 16 
    train_loader = getPotholeDataloader(batch_size, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
    
    rng = jax.random.PRNGKey(42)
    model = PotholeSegmenter()
    dummy_x = jnp.ones((1, 640, 640, 3))
    
    variables = model.init(rng, dummy_x)
    
    total_steps = len(train_loader) * 50 
    lr_schedule = optax.cosine_decay_schedule(init_value=1e-4, decay_steps=total_steps)
    tx = optax.adamw(learning_rate=lr_schedule, weight_decay=1e-4)
    
    state = train_state.TrainState.create(
        apply_fn=model.apply,
        params=variables['params'],
        tx=tx
    )

    #Orbax Checkpointer Setup in Kaggle Working Directory
    checkpoint_dir = os.path.abspath('/kaggle/working/checkpoints')
    checkpointer = ocp.StandardCheckpointer()
    options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
    mngr = ocp.CheckpointManager(checkpoint_dir, checkpointer, options)
    
    epochs = 50
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, (images_np, masks_np) in enumerate(train_loader):
            images = jax.device_put(images_np, data_sharding)
            masks = jax.device_put(masks_np, data_sharding)
            
            state, loss = train_step(state, images, masks)
            epoch_loss += loss
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss:.4f}")
                
        print(f"--- Epoch {epoch+1} Complete. Avg Loss: {epoch_loss / len(train_loader):.4f} ---")
        
        # Save checkpoint at the end of each epoch to /kaggle/working/checkpoints/
        mngr.save(epoch, args=ocp.args.StandardSave(state))
        mngr.wait_until_finished()
        print(f"Checkpoint saved for epoch {epoch+1} at {checkpoint_dir}")

if __name__ == "__main__":
    train_model()
